<a href="https://colab.research.google.com/github/jessicalewinter/deep-learning-specialization/blob/main/course-notebooks/deep-architectures/Lab04_Redes_residuais_aluno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 · Redes residuais

**Arquiteturas Profundas e Representação de Dados** · Especialização em Deep Learning · CIn.AI / CIn-UFPE

No Lab 3, a rede plana de 20 camadas treinou pior que a de 8, e começar as camadas extras como identidade resolveu boa parte do problema. Hoje você constrói o bloco que faz isso de fábrica: y = x + F(x).

- **Parte A (em aula, guiada, ~45 min):** o bloco residual, a ResNet-20, o gradiente por camada na inicialização e um treino lado a lado com a rede plana.
- **Parte B (em casa):** mais fundo, agora com atalho: 20 × 56 camadas, plana × residual.
- **Parte C (em casa):** o estudo de lesão. Tire um bloco de uma ResNet-56 treinada e uma camada de uma VGG-16 treinada.
- **Desafio extra (opcional, não conta para a nota):** ResNet-56 sem BatchNorm, com o ramo começando em zero.

**Antes de começar**

1. *Arquivo → Salvar uma cópia no Drive*.
2. *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*.
3. Rode as células na ordem. As células **Verificação** dizem `OK` ou `ERRO`.
4. No fim, a última célula gera o seu **código de conclusão** para o formulário de entrega.

## Parte A · em aula

### Preparação · configuração, dados e funções prontas

In [ ]:
# Célula 1 · Configuração
import os, time, json, hashlib, random, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = DEVICE == 'cuda'
# Na CPU há menos imagens; mini-batches menores compensam o número menor de passos.
EPOCAS, BS = (5, 128) if DEVICE == 'cuda' else (4, 64)
print('GPU:', torch.cuda.get_device_name(0)) if DEVICE == 'cuda' else print('Sem GPU: o lab usa 10 mil imagens e redes menores na CPU.')

CHECKS = {}
def check(chave, condicao, ok, erro):
    """Registra uma verificação e mostra OK ou ERRO."""
    CHECKS[chave] = bool(condicao)
    print(('OK    ' + ok) if condicao else ('ERRO  ' + erro))

In [ ]:
# Célula 2 · CIFAR-10 (como nos labs anteriores)
ESPELHO_URL = None   # "https://drive.google.com/file/d/1fBXTy06cRH4M4fugrFSQOwqvhjOfAqDS/view?usp=sharing"

t0 = time.time()
if ESPELHO_URL:
    if 'drive.google.com' in ESPELHO_URL:
        import gdown                      # já vem no Colab; baixa arquivos grandes do Google Drive
        gdown.download(ESPELHO_URL, 'cifar10.npz', quiet=False, fuzzy=True)
    else:
        import urllib.request
        urllib.request.urlretrieve(ESPELHO_URL, 'cifar10.npz')
    d = np.load('cifar10.npz')
    tr_x, tr_y, te_x, te_y = d['x_train'], d['y_train'], d['x_test'], d['y_test']
else:
    tr = torchvision.datasets.CIFAR10(root='dados', train=True, download=True)
    te = torchvision.datasets.CIFAR10(root='dados', train=False, download=True)
    tr_x, tr_y, te_x, te_y = tr.data, np.array(tr.targets), te.data, np.array(te.targets)

N_TREINO, N_TESTE = (len(tr_x), len(te_x)) if DEVICE == 'cuda' else (10_000, 2_000)
def para_tensor(x, y, n):
    return torch.from_numpy(x[:n]).permute(0, 3, 1, 2).float().div(255), torch.from_numpy(np.asarray(y[:n])).long()
x_treino, y_treino = para_tensor(tr_x, tr_y, N_TREINO)
x_teste, y_teste = para_tensor(te_x, te_y, N_TESTE)
MEDIA = x_treino.mean(dim=(0, 2, 3), keepdim=True); DESVIO = x_treino.std(dim=(0, 2, 3), keepdim=True)
x_treino = ((x_treino - MEDIA) / DESVIO).to(DEVICE); x_teste = ((x_teste - MEDIA) / DESVIO).to(DEVICE)
y_treino, y_teste = y_treino.to(DEVICE), y_teste.to(DEVICE)
print(f'Treino: {tuple(x_treino.shape)} · Teste: {tuple(x_teste.shape)} · {time.time() - t0:.0f}s')
check('dados', x_treino.shape[1:] == (3, 32, 32) and len(x_treino) == N_TREINO, 'dados carregados', 'formato inesperado: rode a célula 2 de novo')

In [ ]:
# Célula 3 · Funções prontas: avaliar e treinar (as mesmas do Lab 3)
def avaliar(model, x, y, bs=1000):
    model.eval(); acertos = 0
    with torch.no_grad(), torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
        for i in range(0, len(x), bs):
            acertos += (model(x[i:i + bs]).argmax(1) == y[i:i + bs]).sum().item()
    return acertos / len(x)

def treinar(model, epocas=EPOCAS, bs=BS, lr=1e-3, nome='modelo', log_cada=20):
    """Treina com Adam no CIFAR-10. Devolve a curva da perda, o acerto no treino e no teste e o tempo."""
    torch.manual_seed(SEED); model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr); scaler = torch.amp.GradScaler(enabled=USE_AMP)
    t0 = time.time(); curva, acum, passo = [], [], 0
    for ep in range(epocas):
        model.train(); ordem = torch.randperm(len(x_treino), device=DEVICE)
        for i in range(0, len(x_treino), bs):
            b = ordem[i:i + bs]
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
                perda = F.cross_entropy(model(x_treino[b]).float(), y_treino[b])
            opt.zero_grad(set_to_none=True); scaler.scale(perda).backward(); scaler.step(opt); scaler.update()
            acum.append(perda.item()); passo += 1
            if passo % log_cada == 0: curva.append((passo, float(np.mean(acum)))); acum = []
    r = {'modelo': model, 'curva': curva, 'acc_treino': avaliar(model, x_treino[:10_000], y_treino[:10_000]),
         'acc_teste': avaliar(model, x_teste, y_teste), 'tempo': time.time() - t0}
    print(f"{nome}: acerto no treino {r['acc_treino']:.1%} · no teste {r['acc_teste']:.1%} · {r['tempo']:.0f}s")
    return r

def contar(m):
    return sum(p.numel() for p in m.parameters())

### Passo 1 · O bloco residual (10 min)

Um bloco básico da ResNet tem um **ramo** com duas convoluções 3×3 (cada uma seguida de BatchNorm, com ReLU entre elas) e um **atalho**. O atalho é a identidade quando o formato não muda; quando o bloco reduz o mapa (stride 2) e aumenta os canais, o atalho vira uma convolução 1×1 com o mesmo stride (a opção B do artigo).

Na versão **plana**, o bloco tem as mesmas duas convoluções e nenhum atalho: `y = ReLU(F(x))`. Na **residual**: `y = ReLU(F(x) + atalho(x))`. Complete o `forward`.

In [ ]:
# Célula 4 · O bloco
class Bloco(nn.Module):
    def __init__(self, cin, cout, stride=1, residual=True):
        super().__init__()
        self.residual = residual
        self.ramo = nn.Sequential(
            nn.Conv2d(cin, cout, 3, stride, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(),
            nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout))
        self.atalho = nn.Identity()
        if residual and (stride != 1 or cin != cout):
            self.atalho = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, nonlinearity='relu')

    def forward(self, x):
        if not self.residual:
            return F.relu(self.ramo(x))
        # TODO: some a saída do ramo com o atalho da entrada e aplique a ReLU
        raise NotImplementedError('Complete o TODO acima')


**Verificação.** Primeiro o formato; depois, um teste da ideia central: se o último BatchNorm do ramo começar com peso zero, o ramo devolve zero e o bloco copia a entrada.

In [ ]:
_b = Bloco(16, 32, stride=2)
_ok_formato = tuple(_b(torch.randn(2, 16, 32, 32)).shape) == (2, 32, 16, 16)
_b = Bloco(16, 16).eval()
nn.init.zeros_(_b.ramo[4].weight); nn.init.zeros_(_b.ramo[4].bias)   # o ramo passa a devolver zero
_x = torch.rand(2, 16, 8, 8)                                         # entrada positiva, como depois de uma ReLU
_ok_identidade = torch.allclose(_b(_x), _x)
print('formato com stride 2:', _ok_formato, '· ramo zerado copia a entrada:', _ok_identidade)
check('bloco', _ok_formato and _ok_identidade, 'o bloco residual está certo', 'confira o forward: ReLU(ramo(x) + atalho(x))')

### Passo 2 · ResNet-20 e a plana de 20 camadas (5 min)

As duas redes do artigo para o CIFAR-10: uma convolução de entrada, três estágios de `n` blocos (16, 32 e 64 filtros) e a camada de saída. São `6n + 2` camadas com pesos: com `n = 3`, 20 camadas. A única diferença entre elas é o atalho.

In [ ]:
# Célula 5 · As duas redes (prontas)
class Rede(nn.Module):
    def __init__(self, n, residual=True, larg=(16, 32, 64)):
        super().__init__()
        self.entrada = nn.Sequential(nn.Conv2d(3, larg[0], 3, 1, 1, bias=False), nn.BatchNorm2d(larg[0]), nn.ReLU())
        nn.init.kaiming_normal_(self.entrada[0].weight, nonlinearity='relu')
        blocos, cin = [], larg[0]
        for e, c in enumerate(larg):
            for i in range(n):
                blocos.append(Bloco(cin, c, 2 if (e > 0 and i == 0) else 1, residual)); cin = c
        self.blocos = nn.Sequential(*blocos)
        self.saida = nn.Linear(cin, 10)

    def forward(self, x):
        return self.saida(F.adaptive_avg_pool2d(self.blocos(self.entrada(x)), 1).flatten(1))

def camadas_com_pesos(m):
    """Conta as convoluções 3×3 e a camada linear (os atalhos 1×1 não entram na conta, como no artigo)."""
    return sum(isinstance(x, nn.Linear) or (isinstance(x, nn.Conv2d) and x.kernel_size == (3, 3)) for x in m.modules())

for nome, m in [('plana-20', Rede(3, False)), ('ResNet-20', Rede(3, True))]:
    print(f'{nome}: {camadas_com_pesos(m)} camadas, {contar(m):,} parâmetros'.replace(',', '.'))
check('redes', camadas_com_pesos(Rede(3)) == 20 and contar(Rede(3)) == 272_474, 'ResNet-20 com 20 camadas e 272.474 parâmetros', 'a contagem não bate: rode as células 4 e 5 de novo')

### Passo 3 · O gradiente na inicialização (10 min)

Antes de treinar, olhe o tamanho do gradiente que chega em cada uma das 55 convoluções 3×3 de uma rede de 56 camadas, plana e residual, as duas com BatchNorm.

**Preveja antes de rodar:** em qual das duas a primeira camada recebe um gradiente mais parecido com o da última? Responda `'plana'` ou `'residual'`.

In [ ]:
# Célula 6 · Sua previsão
PREVISAO = ''   # 'plana' ou 'residual'
check('previsao', PREVISAO in ('plana', 'residual'), 'previsão registrada', "responda 'plana' ou 'residual'")

In [ ]:
# Célula 7 · Gradiente por camada na inicialização
def gradiente_por_camada(m):
    m = m.to(DEVICE).train(); m.zero_grad()
    F.cross_entropy(m(x_treino[:256]), y_treino[:256]).backward()
    return [c.weight.grad.norm().item() for c in m.modules() if isinstance(c, nn.Conv2d) and c.kernel_size == (3, 3)]

GRAD = {}
for nome, residual in [('plana', False), ('residual', True)]:
    torch.manual_seed(SEED); GRAD[nome] = gradiente_por_camada(Rede(9, residual))
    print(f'{nome:9s} camada 1: {GRAD[nome][0]:.3g} · camada 55: {GRAD[nome][-1]:.3g} · razão: {GRAD[nome][0] / GRAD[nome][-1]:.3g}')
fig, ax = plt.subplots(figsize=(9, 3.4))
for nome, g in GRAD.items(): ax.semilogy(range(1, len(g) + 1), g, 'o-', ms=3, label=nome)
ax.set_xlabel('convolução'); ax.set_ylabel('norma do gradiente'); ax.legend(); ax.grid(alpha=.3); plt.show()
_razao = {k: g[0] / g[-1] for k, g in GRAD.items()}
check('gradiente', _razao['residual'] < _razao['plana'], 'na residual o gradiente da camada 1 fica mais perto do da última',
      'resultado inesperado: rode de novo as células 4 a 7')

Na rede plana, com BatchNorm, o gradiente da primeira camada costuma ser centenas de vezes maior que o da última: a BatchNorm, sozinha, faz o gradiente **explodir** em redes planas fundas (Yang et al., 2019). Com o atalho, a derivada de cada bloco é 1 + ∂F/∂x, e as camadas recebem gradientes de tamanho parecido.

### Passo 4 · Treino lado a lado (15 min)

As duas redes de 20 camadas, com o mesmo treino. Mande o acerto no treino das duas no formulário da turma.

In [ ]:
# Célula 8 · Plana-20 × ResNet-20
RESULTADOS_A = {}
for nome, residual in [('plana-20', False), ('ResNet-20', True)]:
    torch.manual_seed(SEED); RESULTADOS_A[nome] = treinar(Rede(3, residual), nome=nome)
fig, ax = plt.subplots(figsize=(8, 3.4))
for nome, r in RESULTADOS_A.items(): ax.plot(*zip(*r['curva']), label=nome)
ax.set_xlabel('passo'); ax.set_ylabel('perda no treino'); ax.legend(); plt.show()
print(f"Para o formulário: plana-20 {RESULTADOS_A['plana-20']['acc_treino']:.3f} · ResNet-20 {RESULTADOS_A['ResNet-20']['acc_treino']:.3f}")
check('treino_a', len(RESULTADOS_A) == 2, 'as duas redes treinadas', 'rode a célula de novo')

## Parte B · em casa: mais fundo, com atalho

Agora 56 camadas (`n = 9`), plana e residual. Na CPU, a célula usa 32 camadas (`n = 5`) para caber no tempo. Compare o acerto **no treino** com o das redes de 20 camadas da parte A.

In [ ]:
# Célula 9 · 20 × 56 camadas
N_FUNDO = 9 if DEVICE == 'cuda' else 5
RESULTADOS_B = dict(RESULTADOS_A)
for nome, residual in [(f'plana-{6 * N_FUNDO + 2}', False), (f'ResNet-{6 * N_FUNDO + 2}', True)]:
    torch.manual_seed(SEED); RESULTADOS_B[nome] = treinar(Rede(N_FUNDO, residual), nome=nome)
print()
for nome, r in RESULTADOS_B.items():
    print(f"{nome:12s} treino {r['acc_treino']:.1%} · teste {r['acc_teste']:.1%}")
check('tabela_b', len(RESULTADOS_B) == 4, 'quatro redes treinadas', 'rode a célula de novo')

**Responda (3 a 5 linhas):** o que aconteceu com a rede plana ao passar de 20 para mais camadas? E com a residual? Relacione com a degradação do Lab 3 e com o gradiente que você viu no passo 3.

In [ ]:
# Célula 10 · Sua resposta
RESPOSTA_B = """
escreva aqui
"""
_t = RESPOSTA_B.replace('escreva aqui', '').strip()
check('resposta_b', len(_t) >= 120, 'resposta registrada', 'escreva pelo menos 3 linhas')

## Parte C · em casa: tire um bloco

Veit, Wilber & Belongie (2016) tiraram blocos de ResNets treinadas, um de cada vez, sem treinar de novo, e quase nada mudou. Na VGG, tirar qualquer camada derruba a rede. Vamos repetir com duas redes treinadas no CIFAR-10, baixadas do repositório `chenyaofo/pytorch-cifar-models` (licença BSD): uma ResNet-56 (94,4% de acerto) e uma VGG-16 com BatchNorm (94,2%).

Essas redes foram treinadas com outra normalização das imagens; a função `para_pretreinada` converte.

In [ ]:
# Célula 11 · As redes pré-treinadas
r56 = torch.hub.load('chenyaofo/pytorch-cifar-models', 'cifar10_resnet56', pretrained=True, trust_repo=True).to(DEVICE).eval()
vgg = torch.hub.load('chenyaofo/pytorch-cifar-models', 'cifar10_vgg16_bn', pretrained=True, trust_repo=True).to(DEVICE).eval()
MEDIA_PT = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1); DESVIO_PT = torch.tensor([0.2023, 0.1994, 0.2010]).view(1, 3, 1, 1)
def para_pretreinada(x):
    """Desfaz a normalização deste notebook e aplica a das redes pré-treinadas."""
    return (x * DESVIO.to(x.device) + MEDIA.to(x.device) - MEDIA_PT.to(x.device)) / DESVIO_PT.to(x.device)
x_pt = para_pretreinada(x_teste)
BASE = {'ResNet-56': avaliar(r56, x_pt, y_teste), 'VGG-16': avaliar(vgg, x_pt, y_teste)}
print({k: f'{v:.1%}' for k, v in BASE.items()})
print(r56.layer1[0])

A ResNet-56 tem três estágios (`layer1`, `layer2`, `layer3`) de 9 blocos. Cada bloco calcula `ReLU(ramo(x) + atalho(x))`, em que o atalho é `downsample` quando existe (nos blocos que reduzem o mapa) e a identidade nos outros.

Complete `SemRamo`: um bloco que joga fora o ramo e fica só com o atalho.

In [ ]:
# Célula 12 · Remover um bloco
class SemRamo(nn.Module):
    def __init__(self, bloco):
        super().__init__()
        self.downsample = bloco.downsample          # None nos blocos que não mudam o formato
    def forward(self, x):
        # TODO: devolva ReLU(atalho(x)); o atalho é self.downsample(x) se ele existir, senão o próprio x
        raise NotImplementedError('Complete o TODO acima')

LESOES_RES = []
for estagio in ['layer1', 'layer2', 'layer3']:
    for i in range(9):
        m = copy.deepcopy(r56); getattr(m, estagio)[i] = SemRamo(getattr(m, estagio)[i])
        LESOES_RES.append((f'{estagio}.{i}', avaliar(m, x_pt, y_teste)))
LESOES_VGG = []
f = list(vgg.features)
for k, i in enumerate([j for j, mm in enumerate(f) if isinstance(mm, nn.Conv2d)]):
    if f[i].in_channels == f[i].out_channels:          # só dá para tirar camadas que não mudam os canais
        m = copy.deepcopy(vgg); m.features[i] = nn.Identity(); m.features[i + 1] = nn.Identity()
        LESOES_VGG.append((f'conv {k + 1}', avaliar(m, x_pt, y_teste)))

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(range(27), [a for _, a in LESOES_RES], color='#3C78D8', label='ResNet-56 sem um bloco')
ax.bar(range(29, 29 + len(LESOES_VGG)), [a for _, a in LESOES_VGG], color='#DB1E2F', label='VGG-16 sem uma camada')
ax.axhline(BASE['ResNet-56'], c='#3C78D8', ls='--'); ax.axhline(BASE['VGG-16'], c='#DB1E2F', ls='--'); ax.axhline(0.1, c='gray', ls=':')
ax.set_ylabel('acerto no teste'); ax.set_xticks([]); ax.legend(); plt.show()
QUEDA_MEDIANA = float(np.median([BASE['ResNet-56'] - a for _, a in LESOES_RES]))
print(f'Queda mediana ao tirar um bloco da ResNet-56: {QUEDA_MEDIANA:.1%}')
print('VGG-16 sem uma camada:', ', '.join(f'{n}: {a:.0%}' for n, a in LESOES_VGG))

**Verificação.**

In [ ]:
check('lesao', len(LESOES_RES) == 27 and QUEDA_MEDIANA < 0.05 and max(a for _, a in LESOES_VGG) < 0.3,
      'a ResNet quase não sente a falta de um bloco; a VGG vira chute', 'confira o SemRamo: ReLU(downsample(x)) ou ReLU(x)')

**Responda (3 a 5 linhas):** por que a ResNet tolera perder um bloco e a VGG não? Use a ideia de caminhos: quantos caminhos ligam a entrada à saída em cada rede, e quantos sobram depois de uma remoção? Por que os blocos que reduzem o mapa doem mais?

In [ ]:
# Célula 13 · Sua resposta
RESPOSTA_C = """
escreva aqui
"""
_t = RESPOSTA_C.replace('escreva aqui', '').strip()
check('resposta_c', len(_t) >= 120, 'resposta registrada', 'escreva pelo menos 3 linhas')

## Desafio extra (opcional) · ResNet sem BatchNorm

Sem BatchNorm, somar a entrada a cada bloco faz o sinal crescer bloco a bloco. Uma saída é começar o ramo em zero, para cada bloco nascer como identidade: é o `zero_init_residual` do torchvision. Monte a versão sem BatchNorm (troque `nn.BatchNorm2d` por `nn.Identity`), zere a última convolução de cada ramo e compare com a mesma rede inicializada com He.

In [ ]:
# Célula 14 · Sem BatchNorm (opcional)
FAZER_DESAFIO = False
if FAZER_DESAFIO:
    pass   # seu código aqui